In [1]:
from openai import OpenAI
client = OpenAI(
  api_key = "sk-281201073d5d402197cb600119e8433c",
   base_url = "https://api.deepseek.com/v1"
)

In [41]:
# 文件型的关系型数据库
import sqlite3
conn = sqlite3.connect("test.db")
# 创建一个游标对象
# 执行sql 操作
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS
employees(
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")
sample_data = [
    (6, "黄佳", "销售", 50000),
    (7, "宁宁", "工程", 75000),
    (8, "芊芊", "销售", 60000),
    (9, "悦悦", "工程", 80000),
    (10, "黄仁勋", "市场", 55000)
]
cursor.executemany("INSERT INTO employees VALUES(?,?,?,?)",sample_data)
conn.commit()

In [42]:
# 获取数据库Schema
# Schema 应详细描述每个表的字段和类型，
# 这有助于 Claude 理解表的结构和关联。
# SQLite命令，查看employees表的列名、类型等结构信息。
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
print(schema)
# 列表推导式
# 使用 f-string（格式化字符串字面量），将列名和类型拼接成一个字符串。
schema_str = "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print("数据库Schema:")
print(schema_str)

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'name', 'TEXT', 0, None, 0), (2, 'department', 'TEXT', 0, None, 0), (3, 'salary', 'INTEGER', 0, None, 0)]
数据库Schema:
CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


In [43]:
# 销售部门平均工资多少？先分组再算平均
# text2sql 数据库平权
# vibe coding 平权了代码开发
def ask_deepseek(query, schema):
    # 模板
    prompt = f"""
    这是一个数据库的Schema：
    {schema}
    根据这个Schema，请输出一个SQL查询来回答以下问题。
    只输出SQL 查询语句本身，不要使用任何markdown格式，
    不要包含反引号，代码块标记或额外说明。
    问题：{query}
    """
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        max_tokens=2048,
        messages=[{
            "role":"user",
            "content":prompt
        }]
    )
    return response.choices[0].message.content

In [44]:
question = "工程部门员工的姓名和工资是多少"
sql_query = ask_deepseek(question,schema_str)
print(sql_query)

SELECT name, salary FROM EMPLOYEES WHERE department = 'Engineering';


In [45]:
result = cursor.execute(sql_query).fetchall()
for row in result:
    print(row)

In [50]:
question = "在销售部门增加一个新员工，姓名为张三，工资为45000"
sql_query = ask_deepseek(question,schema_str);
cursor.execute(sql_query)
conn.commit()

In [51]:
question = "将王二的工资调整为55000"
sql_query = ask_deepseek(question,schema_str)
print(sql_query)
cursor.execute(sql_query)
conn.commit()

UPDATE EMPLOYEES SET salary = 55000 WHERE name = '王二';


In [52]:
question = "删除市场部门的王二"
sql_query = ask_deepseek(question,schema_str)
print(sql_query)
cursor.execute(sql_query)
conn.commit()

DELETE FROM EMPLOYEES WHERE name = '王二' AND department = '市场部';


In [59]:
question = "查询所有员工的信息"
sql_query = ask_deepseek(question,schema_str)
print(sql_query)
result = cursor.execute(sql_query).fetchall()
for row in result:
    print(row)

SELECT * FROM EMPLOYEES;
(6, '黄佳', '销售', 50000)
(7, '宁宁', '工程', 75000)
(8, '芊芊', '销售', 60000)
(9, '悦悦', '工程', 80000)
(10, '黄仁勋', '市场', 55000)
(11, '张三', '销售', 45000)


In [58]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS departments(
id INTEGER PRIMARY KEY,
name TEXT,
manager TEXT
)
""")
sample_departments = [
    (1,"销售","王经理"),
    (2,"工程","李经理"),
    (3,"市场","张经理")
]
cursor.executemany(
"INSERT INTO departments VALUES(?,?,?)",
    sample_departments
)
conn.commit()

In [63]:
question = "查询departments 表所有的记录，输出所有字段"
sql_query = ask_deepseek(question,schema_str)
print(sql_query)
results = cursor.execute(sql_query).fetchall()

for row in results:
    print(row)

SELECT * FROM departments;
(1, '销售', '王经理')
(2, '工程', '李经理')
(3, '市场', '张经理')


In [64]:
# 获取完整的数据库schema
tables = ["employees", "departments"]
schema_str = ""
for table in tables:
    schema = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    schema_str += f"CREATE TABLE {table} (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n);\n\n"

print("完整的数据库schema:")
print(schema_str)

完整的数据库schema:
CREATE TABLE employees (
id INTEGER
name TEXT
department TEXT
salary INTEGER
);

CREATE TABLE departments (
id INTEGER
name TEXT
manager TEXT
);




In [65]:
question = "根据两个表之间的关系，列出每个部门的员工人数和平均工资"
sql_query = ask_deepseek(question,schema_str)
# 右链接，以department 表为主
# Group By 分组
# COUNT AVG
print(sql_query)

SELECT d.name AS department, COUNT(e.id) AS employee_count, AVG(e.salary) AS average_salary
FROM departments d
LEFT JOIN employees e ON e.department = d.name
GROUP BY d.id, d.name;
